In [1]:
from dotenv import load_dotenv
import os

load_dotenv()


True

# Document Loader

In [2]:
file_path = "data\LLM-Engineers-Handbook.pdf"

In [3]:
from pathlib import Path
from typing import List, Dict, Any, Tuple
import pymupdf
from docx import Document as DocxDocument
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from datetime import datetime
import hashlib
import re
import unicodedata

class OptimizedPreprocessedLoader:
    """RAG loader with page numbers and optimized chunk sizes"""

    def __init__(self, chunk_size=1000, chunk_overlap=200,
                 min_chunk_size=None, max_chunk_size=None,
                 preprocessing_config=None):

        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

        # Set min/max chunk sizes (default: 80-120% of target)
        self.min_chunk_size = min_chunk_size or int(chunk_size * 0.8)
        self.max_chunk_size = max_chunk_size or int(chunk_size * 1.2)

        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len,
        )

        # Default preprocessing config
        self.config = {
            'remove_extra_whitespace': True,
            'remove_extra_newlines': True,
            'normalize_unicode': True,
            'fix_encoding_errors': True,
            'fix_common_ocr_errors': True,
            'remove_headers_footers': True,
            'remove_page_numbers': True,
            'remove_short_lines': True,
        }

        if preprocessing_config:
            self.config.update(preprocessing_config)

    def load_and_split(self, source: str, custom_metadata: Dict = None) -> List[Document]:
        """Load with page tracking and optimized chunking"""

        if source.startswith(('http://', 'https://')):
            return self._load_from_url(source, custom_metadata or {})
        else:
            return self._load_from_file(source, custom_metadata or {})

    def _load_from_url(self, url: str, custom_metadata: Dict) -> List[Document]:
        """Load URL (no page numbers for web content)"""
        try:
            loader = WebBaseLoader(
                web_paths=[url]
            )
            docs = loader.load()

            # Preprocess text
            for doc in docs:
                doc.page_content = self.preprocess_text(doc.page_content)
                doc.metadata.update({
                    'source': url,
                    'source_type': 'url',
                    'file_type': 'html',
                    'loaded_at': datetime.now().isoformat(),
                    'doc_id': self._generate_doc_id(url),
                    **custom_metadata
                })

            # Optimized chunking
            chunks = self._optimized_split(docs)

            return chunks

        except Exception as e:
            raise Exception(f"Failed to load URL: {e}")

    def _load_from_file(self, file_path: str, custom_metadata: Dict) -> List[Document]:
        """Load file with page numbers and optimized chunking"""
        path = Path(file_path)

        if not path.exists():
            raise FileNotFoundError(f"File not found: {file_path}")

        suffix = path.suffix.lower()

        if suffix == '.pdf':
            # Extract with page-by-page tracking
            page_contents, file_metadata = self._extract_pdf_with_pages(path)
        elif suffix == '.docx':
            # DOCX doesn't have reliable page numbers, use sections
            page_contents, file_metadata = self._extract_docx_with_sections(path)
        elif suffix in ['.html', '.htm']:
            text, file_metadata = self._extract_html_with_metadata(path)
            page_contents = [{'page_num': 1, 'text': text}]
        else:
            raise ValueError(f"Unsupported format: {suffix}")

        # Combine metadata
        base_metadata = {
            'source': str(path.absolute()),
            'source_type': 'file',
            'file_type': suffix[1:],
            'file_name': path.name,
            'file_size': path.stat().st_size,
            'loaded_at': datetime.now().isoformat(),
            'doc_id': self._generate_doc_id(str(path)),
            **file_metadata,
            **custom_metadata
        }

        # Create documents with page tracking
        docs_with_pages = []
        for page_info in page_contents:
            preprocessed_text = self.preprocess_text(page_info['text'])

            if preprocessed_text.strip():  # Skip empty pages
                doc = Document(
                    page_content=preprocessed_text,
                    metadata={
                        **base_metadata,
                        'page_number': page_info['page_num'],
                        'page_start': page_info['page_num'],
                        'page_end': page_info['page_num'],
                    }
                )
                docs_with_pages.append(doc)

        # Optimized chunking with page tracking
        chunks = self._optimized_split_with_pages(docs_with_pages)

        return chunks

    def _extract_pdf_with_pages(self, path: Path) -> Tuple[List[Dict], Dict]:
        """Extract PDF page-by-page"""
        doc = pymupdf.open(path)

        page_contents = []
        for page_num, page in enumerate(doc, start=1):
            page_text = page.get_text()
            if page_text.strip():
                page_contents.append({
                    'page_num': page_num,
                    'text': page_text
                })

        metadata = {
            'page_count': len(doc),
            'author': doc.metadata.get('author', ''),
            'title': doc.metadata.get('title', ''),
            'subject': doc.metadata.get('subject', ''),
            'keywords': doc.metadata.get('keywords', ''),
            'creator': doc.metadata.get('creator', ''),
            'creation_date': doc.metadata.get('creationDate', ''),
            'modification_date': doc.metadata.get('modDate', ''),
        }

        doc.close()
        metadata = {k: v for k, v in metadata.items() if v}

        return page_contents, metadata

    def _extract_docx_with_sections(self, path: Path) -> Tuple[List[Dict], Dict]:
        """Extract DOCX by sections (paragraphs grouped by ~1 page worth)"""
        doc = DocxDocument(path)

        # Estimate: ~500 words per page
        WORDS_PER_PAGE = 500

        page_contents = []
        current_page_text = []
        current_word_count = 0
        page_num = 1

        for para in doc.paragraphs:
            if para.text.strip():
                para_words = len(para.text.split())
                current_page_text.append(para.text)
                current_word_count += para_words

                # When we reach ~page worth, create a new page
                if current_word_count >= WORDS_PER_PAGE:
                    page_contents.append({
                        'page_num': page_num,
                        'text': '\n\n'.join(current_page_text)
                    })
                    current_page_text = []
                    current_word_count = 0
                    page_num += 1

        # Add remaining text
        if current_page_text:
            page_contents.append({
                'page_num': page_num,
                'text': '\n\n'.join(current_page_text)
            })

        # Extract tables as separate "pages"
        for table in doc.tables:
            page_num += 1
            table_text = []
            for row in table.rows:
                row_text = ' | '.join(cell.text.strip() for cell in row.cells)
                if row_text.strip():
                    table_text.append(row_text)

            if table_text:
                page_contents.append({
                    'page_num': page_num,
                    'text': '\n'.join(table_text)
                })

        core_props = doc.core_properties
        metadata = {
            'author': core_props.author or '',
            'title': core_props.title or '',
            'subject': core_props.subject or '',
            'keywords': core_props.keywords or '',
            'created': core_props.created.isoformat() if core_props.created else '',
            'modified': core_props.modified.isoformat() if core_props.modified else '',
            'paragraph_count': len(doc.paragraphs),
            'table_count': len(doc.tables),
            'estimated_pages': len(page_contents),
        }

        metadata = {k: v for k, v in metadata.items() if v}

        return page_contents, metadata

    def _extract_html_with_metadata(self, path: Path) -> Tuple[str, Dict]:
        """Extract HTML text + metadata"""
        from bs4 import BeautifulSoup

        with open(path, 'r', encoding='utf-8') as f:
            html_content = f.read()

        soup = BeautifulSoup(html_content, 'html.parser')

        metadata = {
            'title': soup.title.string if soup.title else '',
            'description': '',
            'keywords': '',
            'author': '',
        }

        for meta in soup.find_all('meta'):
            name = meta.get('name', '').lower()
            content = meta.get('content', '')

            if name == 'description':
                metadata['description'] = content
            elif name == 'keywords':
                metadata['keywords'] = content
            elif name == 'author':
                metadata['author'] = content

        for tag in soup(['script', 'style', 'nav', 'footer', 'header']):
            tag.decompose()

        text = soup.get_text(separator='\n', strip=True)
        metadata = {k: v for k, v in metadata.items() if v}

        return text, metadata

    def _optimized_split(self, docs: List[Document]) -> List[Document]:
        """Split and optimize chunk sizes (for web content without pages)"""
        # Initial split
        chunks = self.splitter.split_documents(docs)

        # Optimize chunk sizes
        optimized_chunks = self._merge_small_and_split_large(chunks)

        # Add chunk metadata
        for i, chunk in enumerate(optimized_chunks):
            chunk.metadata['chunk_index'] = i
            chunk.metadata['total_chunks'] = len(optimized_chunks)
            chunk.metadata['chunk_id'] = f"{chunk.metadata.get('doc_id', 'unknown')}_chunk_{i}"
            chunk.metadata['chunk_size'] = len(chunk.page_content)

        return optimized_chunks

    def _optimized_split_with_pages(self, docs_with_pages: List[Document]) -> List[Document]:
        """Split and optimize chunk sizes while preserving page numbers"""
        # Initial split
        chunks = self.splitter.split_documents(docs_with_pages)

        # Optimize chunk sizes
        optimized_chunks = self._merge_small_and_split_large(chunks)

        # Add chunk metadata with page tracking
        for i, chunk in enumerate(optimized_chunks):
            chunk.metadata['chunk_index'] = i
            chunk.metadata['total_chunks'] = len(optimized_chunks)
            chunk.metadata['chunk_id'] = f"{chunk.metadata.get('doc_id', 'unknown')}_chunk_{i}"
            chunk.metadata['chunk_size'] = len(chunk.page_content)

            # Page numbers already preserved from original docs
            # If chunk spans multiple pages, page_start and page_end will show range

        return optimized_chunks

    def _merge_small_and_split_large(self, chunks: List[Document]) -> List[Document]:
        """Ensure all chunks are near target size"""
        optimized = []
        i = 0

        while i < len(chunks):
            current_chunk = chunks[i]
            current_size = len(current_chunk.page_content)

            # Case 1: Chunk is too small - try to merge with next
            if current_size < self.min_chunk_size and i < len(chunks) - 1:
                next_chunk = chunks[i + 1]

                # Check if we can merge without exceeding max
                combined_text = current_chunk.page_content + "\n\n" + next_chunk.page_content
                combined_size = len(combined_text)

                if combined_size <= self.max_chunk_size:
                    # Merge chunks
                    merged_metadata = current_chunk.metadata.copy()

                    # Update page range if both have page numbers
                    if 'page_number' in current_chunk.metadata and 'page_number' in next_chunk.metadata:
                        merged_metadata['page_start'] = current_chunk.metadata.get('page_start', current_chunk.metadata['page_number'])
                        merged_metadata['page_end'] = next_chunk.metadata.get('page_end', next_chunk.metadata['page_number'])
                        merged_metadata['page_number'] = f"{merged_metadata['page_start']}-{merged_metadata['page_end']}"

                    merged_chunk = Document(
                        page_content=combined_text,
                        metadata=merged_metadata
                    )
                    optimized.append(merged_chunk)
                    i += 2  # Skip both chunks
                else:
                    # Can't merge, keep as is
                    optimized.append(current_chunk)
                    i += 1

            # Case 2: Chunk is too large - split it
            elif current_size > self.max_chunk_size:
                # Re-split this chunk
                sub_chunks = self._split_large_chunk(current_chunk)
                optimized.extend(sub_chunks)
                i += 1

            # Case 3: Chunk is in acceptable range
            else:
                optimized.append(current_chunk)
                i += 1

        return optimized

    def _split_large_chunk(self, chunk: Document) -> List[Document]:
        """Split a large chunk into smaller ones"""
        # Use a temporary splitter with smaller size
        temp_splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
        )

        sub_chunks = temp_splitter.split_documents([chunk])

        # Preserve original metadata
        for sub_chunk in sub_chunks:
            sub_chunk.metadata = chunk.metadata.copy()

        return sub_chunks

    def preprocess_text(self, text: str) -> str:
        """Comprehensive text preprocessing"""
        if not text:
            return ""

        if self.config['fix_encoding_errors']:
            text = self._fix_encoding_errors(text)

        if self.config['normalize_unicode']:
            text = unicodedata.normalize('NFKC', text)

        if self.config['fix_common_ocr_errors']:
            text = self._fix_ocr_errors(text)

        if self.config['remove_headers_footers']:
            text = self._remove_headers_footers(text)

        if self.config['remove_page_numbers']:
            text = self._remove_page_numbers(text)

        if self.config['remove_extra_whitespace']:
            text = re.sub(r'[ \t]+', ' ', text)

        if self.config['remove_extra_newlines']:
            text = re.sub(r'\n{3,}', '\n\n', text)

        if self.config['remove_short_lines']:
            lines = text.split('\n')
            lines = [line for line in lines if len(line.split()) >= 3 or line.strip() == '']
            text = '\n'.join(lines)

        return text.strip()

    def _fix_encoding_errors(self, text: str) -> str:
        """Fix common encoding issues"""
        replacements = {
            'â€™': "'", 'â€œ': '"', 'â€': '"',
            'â€"': '—', 'â€"': '–', 'Â': '',
            '\x00': '', '\ufffd': '',
        }
        for old, new in replacements.items():
            text = text.replace(old, new)
        return text

    def _fix_ocr_errors(self, text: str) -> str:
        """Fix common OCR errors"""
        text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)
        return text

    def _remove_headers_footers(self, text: str) -> str:
        """Remove common header/footer patterns"""
        lines = text.split('\n')
        cleaned = []

        for line in lines:
            line_lower = line.lower().strip()
            skip_patterns = [
                r'^page \d+', r'^\d+ of \d+$', r'^chapter \d+',
                r'^confidential', r'^\d+$',
            ]
            if not any(re.match(p, line_lower) for p in skip_patterns):
                cleaned.append(line)

        return '\n'.join(cleaned)

    def _remove_page_numbers(self, text: str) -> str:
        """Remove standalone page numbers"""
        text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
        text = re.sub(r'\bPage\s+\d+\b', '', text, flags=re.IGNORECASE)
        return text

    def _generate_doc_id(self, source: str) -> str:
        """Generate unique document ID"""
        return hashlib.md5(source.encode()).hexdigest()[:16]


c:\Users\hetba\OneDrive\Desktop\Work\Learn\Projects\AllinOneRAG\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
# Usage Examples

# Example 1: Basic usage with page numbers
loader = OptimizedPreprocessedLoader(
    chunk_size=1000,
    chunk_overlap=200,
    min_chunk_size=800,   # 80% of target
    max_chunk_size=1200   # 120% of target
)

chunks = loader.load_and_split(
    r"data\LLM-Engineers-Handbook.pdf"
)


# Embedding 

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

c:\Users\hetba\OneDrive\Desktop\Work\Learn\Projects\AllinOneRAG\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Vectore store

In [4]:
collection_name = "docs"
PERSIST_DIR = "./chroma_db"

In [5]:
from langchain_chroma import Chroma
vectordb = Chroma(
    persist_directory=PERSIST_DIR,
    embedding_function=embedding
)

In [6]:
existing_collections = [
    c.name for c in vectordb._client.list_collections()
]

print(existing_collections)

['langchain', 'docs']


In [7]:
if collection_name in existing_collections:
    print("Collection exists — loading embeddings")

    vectordb = Chroma(
        persist_directory=PERSIST_DIR,
        embedding_function=embedding,
        collection_name=collection_name
    )

else:
    print("Collection not found — creating embeddings")

    vectordb = Chroma.from_documents(
        documents=chunks,
        embedding=embedding,
        persist_directory=PERSIST_DIR,
        collection_name=collection_name
    )


Collection exists — loading embeddings


In [22]:

def get_context(question):

    results = vectordb.similarity_search_with_score(
        question,
        k=5
    )

    context = "\n\n".join([f"{r[0].page_content}\n\nMetadata: {r[0].metadata}" for r in results])

    return  context

# Generation

In [12]:
from langchain_groq import ChatGroq


llm = ChatGroq(model="openai/gpt-oss-120b")

In [13]:
# context = "\n\n".join([f"{r[0].page_content}\n\nMetadata: {r[0].metadata}" for r in results])

In [14]:



def get_llm_response(question, context):
   response = llm.invoke(f"""You are a document-grounded AI assistant.

   Answer the user's question using ONLY the information provided in <context>.

   Strict rules:
   1. Do NOT use outside knowledge.
   2. Do NOT hallucinate.
   3. If the answer is not present, respond:
      "I don't know based on the provided documents."
   4. Every factual statement MUST have a citation.
   5. Citations must reference chunk metadata (source + chunk_id).
   6. If multiple chunks support a fact, include multiple citations.
   7. Keep answers concise and factual.

   Return output in this format:

   Answer:
   <your answer here>

   Citations:
   - source: <source>, chunk_id: <chunk_id>
   - source: <source>, chunk_id: <chunk_id>

   <context>
   {context}
   </context>

   User Question:
   {question}
   """)

   return response

In [15]:
question = "explain vanilla RAG framework in detail and also explain cosine distance formula and generation pipline code."

In [16]:
context = get_context(question)
response = get_llm_response(question, context)

In [21]:
print(context)

how it works. We will then walk you through all the components of a naïve RAG system: chunking, embedding, and vector DBs. Ultimately, we will present various optimizations used for an 
advanced RAG system. Then, we will continue exploring LLM Twin’s RAG feature pipeline architecture. At this step, we will apply all the theoretical aspects we discussed at the beginning of the 
chapter. Finally, we will go through a practical example by implementing the LLM Twin’s RAG 
feature pipeline based on the system design described throughout the book.
The main sections of this chapter are:
An overview of advanced RAG
Exploring the LLM Twin’s RAG feature pipeline architecture
Implementing the LLM Twin’s RAG feature pipeline
By the end of this chapter, you will have a clear and comprehensive understanding of what RAG 
is and how it is applied to our LLM Twin use case.

Metadata: {'chunk_id': '640a89855672abdf_chunk_242', 'doc_id': '640a89855672abdf', 'loaded_at': '2026-02-16T13:23:40.199578', 'fil

In [20]:
print(response.content)

Answer:
I don't know based on the provided documents.

Citations:
- source: c:\Users\hetba\OneDrive\Desktop\Work\Learn\Projects\AllinOneRAG\data\LLM-Engineers-Handbook.pdf, chunk_id: 12
- source: c:\Users\hetba\OneDrive\Desktop\Work\Learn\Projects\AllinOneRAG\data\LLM-Engineers-Handbook.pdf, chunk_id: 241


# Evaluation

In [23]:
import json

with open('ragas_evaluation_qa.json', 'r') as f:
    data = json.load(f)

In [26]:
"""
Ragas Evaluation with Rate Limiting for Groq API
Fixes deprecation warnings and handles rate limits
"""

import time
import json
import os
from datasets import Dataset

# Updated imports to fix deprecation warnings
from ragas.metrics import (
    faithfulness,
    # answer_relevancy,
    context_recall,
    context_precision,
    # answer_correctness,
)
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper



# ============================================================================
# CONFIGURATION
# ============================================================================

# Rate limiting configuration
REQUESTS_PER_MINUTE = 1  # Groq free tier limit
DELAY_BETWEEN_REQUESTS = 60 / REQUESTS_PER_MINUTE  # ~2 seconds
DELAY_BETWEEN_BATCHES = 60  # Extra delay every N questions

# Batch processing configuration
BATCH_SIZE = 2  # Process 5 questions, then pause


# ============================================================================
# INITIALIZE GROQ LLM (Updated method - no deprecation)
# ============================================================================

ragas_llm = LangchainLLMWrapper(llm)


# ============================================================================
# LOAD DATA
# ============================================================================

with open('ragas_evaluation_qa.json', 'r') as f:
    data = json.load(f)

evaluation_data = {
    'user_input': [],
    'response': [],
    'retrieved_contexts': [],
    'reference': []
}


# ============================================================================
# PROCESS QUESTIONS WITH RATE LIMITING
# ============================================================================

print("Processing questions with rate limiting...")
print(f"Delay between requests: {DELAY_BETWEEN_REQUESTS:.2f}s")
print(f"Batch size: {BATCH_SIZE} questions")
print(f"Extra delay between batches: {DELAY_BETWEEN_BATCHES}s\n")

total_questions = len(data['evaluation_dataset'])

for idx, item in enumerate(data['evaluation_dataset'], 1):
    question = item['question']
    ground_truth = item['ground_truth']

    print(f"[{idx}/{total_questions}] Processing: {question[:60]}...")

    # YOUR RAG PIPELINE WITH RETRY LOGIC
    max_retries = 3
    retry_count = 0

    while retry_count < max_retries:
        try:
            # Add delay before each API call
            if idx > 1:  # Don't delay the first request
                time.sleep(DELAY_BETWEEN_REQUESTS)

            # Get context and response from your RAG pipeline
            context = get_context(question)
            response = get_llm_response(question, context)

            print("="*50)
            print("Question:", question)
            print("Context:", context)
            print("Answer:", response.content)
            print("="*50)


            # Success - break the retry loop
            break

        except Exception as e:
            retry_count += 1
            if "rate_limit" in str(e).lower() or "429" in str(e):
                wait_time = 60 * retry_count  # Exponential backoff
                print(f"  ⚠ Rate limit hit. Waiting {wait_time}s before retry {retry_count}/{max_retries}...")
                time.sleep(wait_time)
            else:
                print(f"  ⚠ Error: {str(e)}")
                if retry_count >= max_retries:
                    print(f"  ❌ Failed after {max_retries} retries. Skipping question.")
                    # Use placeholder data if all retries fail
                    context = ["Error: Could not retrieve context"]
                    response = type('obj', (object,), {'content': 'Error: Could not generate response'})()
                    break
                time.sleep(5)

    # CRITICAL FIX: Ensure context is a LIST of strings
    if isinstance(context, str):
        context = [context]
    elif not isinstance(context, list):
        context = [str(context)]

    context = [str(ctx) if not isinstance(ctx, str) else ctx for ctx in context]

    # Append to evaluation data
    evaluation_data['user_input'].append(question)
    evaluation_data['response'].append(str(response.content))
    evaluation_data['retrieved_contexts'].append(context)
    evaluation_data['reference'].append(ground_truth)

    # Add extra delay after every batch
    if idx % BATCH_SIZE == 0 and idx < total_questions:
        print(f"  💤 Batch complete. Resting for {DELAY_BETWEEN_BATCHES}s...\n")
        time.sleep(DELAY_BETWEEN_BATCHES)

print(f"\n✓ Processed all {total_questions} questions successfully!\n")


# ============================================================================
# CREATE DATASET AND EVALUATE
# ============================================================================

dataset = Dataset.from_dict(evaluation_data)

print("Sample data format:")
print(f"Question: {dataset[0]['user_input'][:60]}...")
print(f"Answer: {dataset[0]['response'][:60]}...")
print(f"Contexts type: {type(dataset[0]['retrieved_contexts'])}")
print(f"Number of contexts: {len(dataset[0]['retrieved_contexts'])}")
print()

print("Starting Ragas evaluation...")
print("Note: This will also make API calls and may take time due to rate limits.\n")

# Evaluate with rate-limited execution
results = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        # answer_relevancy,
        context_recall,
        context_precision,
        # answer_correctness
    ],
    llm=ragas_llm,
    raise_exceptions=False,  # Continue even if some evaluations fail
)


C:\Users\hetba\AppData\Local\Temp\ipykernel_3572\2669362902.py:12: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\hetba\AppData\Local\Temp\ipykernel_3572\2669362902.py:12: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import (
C:\Users\hetba\AppData\Local\Temp\ipykernel_3572\2669362902.py:12: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
C:\Users\hetba\AppData\Local\Temp\ipykernel_3572\2

Processing questions with rate limiting...
Delay between requests: 60.00s
Batch size: 2 questions
Extra delay between batches: 60s

[1/10] Processing: What are the three core pipelines in the FTI (Feature/Traini...
Question: What are the three core pipelines in the FTI (Feature/Training/Inference) architecture and what does each one do?
Context: This is powerful, as we can clearly define the scope and interface of each pipeline. Also, it’s easier 
to understand how the three components interact. Ultimately, we have just three instead of 20 
moving pieces, as suggested in Figure 1.4, which is much easier to work with and define.
As shown in Figure 1.5, we have the feature, training, and inference pipelines. We will zoom in on 
each of them and understand their scope and interface.
Figure 1.5: FTI pipelines architecture

Metadata: {'chunk_id': '640a89855672abdf_chunk_70', 'total_chunks': 1101, 'chunk_index': 70, 'creation_date': "D:20241017164514+05'30'", 'loaded_at': '2026-02-16T13:23:4

Evaluating:  23%|██▎       | 7/30 [03:00<10:20, 26.98s/it]Exception raised in Job[7]: TimeoutError()
Exception raised in Job[6]: TimeoutError()
Exception raised in Job[11]: TimeoutError()
Exception raised in Job[10]: TimeoutError()
Exception raised in Job[12]: TimeoutError()
Exception raised in Job[15]: TimeoutError()
Exception raised in Job[13]: TimeoutError()
Exception raised in Job[9]: TimeoutError()
Exception raised in Job[8]: TimeoutError()
Exception raised in Job[16]: TimeoutError()
Evaluating:  93%|█████████▎| 28/30 [05:56<00:17,  8.98s/it]Exception raised in Job[23]: TimeoutError()
Exception raised in Job[25]: TimeoutError()
Evaluating: 100%|██████████| 30/30 [06:00<00:00, 12.02s/it]


In [ ]:
from ragas import evaluate
import pandas as pd
import time

all_results = []

print("Starting SAFE sequential RAGAS evaluation...\n")

for i in range(len(dataset)):
    print(f"Evaluating sample {i+1}/{len(dataset)}")

    single = Dataset.from_dict({
        "user_input": [dataset[i]["user_input"]],
        "response": [dataset[i]["response"]],
        "retrieved_contexts": [dataset[i]["retrieved_contexts"]],
        "reference": [dataset[i]["reference"]],
    })

    try:
        result = evaluate(
            dataset=single,
            metrics=[
                faithfulness,
                # context_recall,
                # context_precision,
            ],
            llm=ragas_llm,
            raise_exceptions=False,
        )

        all_results.append(result.to_pandas())

    except Exception as e:
        print("Evaluation failed:", e)

    print("Sleeping 30s for Groq...")
    time.sleep(30)   # CRITICAL


final_df = pd.concat(all_results)
print(final_df)

Starting SAFE sequential RAGAS evaluation...

Evaluating sample 1/10


Evaluating: 100%|██████████| 1/1 [01:47<00:00, 107.44s/it]


Sleeping 30s for Groq...
Evaluating sample 2/10


Evaluating: 100%|██████████| 1/1 [03:00<00:00, 180.01s/it]


Sleeping 30s for Groq...
Evaluating sample 3/10


Evaluating: 100%|██████████| 1/1 [03:01<00:00, 181.20s/it]


Sleeping 30s for Groq...
Evaluating sample 4/10


Evaluating: 100%|██████████| 1/1 [01:23<00:00, 83.91s/it]


Sleeping 30s for Groq...


In [27]:
# ============================================================================
# DISPLAY RESULTS
# ============================================================================

print("\n" + "="*70)
print(" "*25 + "RAGAS EVALUATION RESULTS")
print("="*70)

# Handle NaN values in results
def safe_score(score):
    """Return score if valid, otherwise return 'N/A'"""
    try:
        if isinstance(score, (int, float)) and not (score != score):  # Check for NaN
            return f"{score}"
        return "N/A"
    except:
        return "N/A"

print(f"\nOverall Scores:")
print(f"  Faithfulness:        {(results['faithfulness'])}")
print(f"  Context Precision:   {(results['context_precision'])}")
print(f"  Context Recall:      {(results['context_recall'])}")
print("\n" + "="*70)

# Convert to DataFrame and display
results_df = results.to_pandas()

# Filter out NaN columns for display
display_cols = ['user_input', 'faithfulness',
                'context_precision', 'context_recall']
available_cols = [col for col in display_cols if col in results_df.columns]

print("\nDetailed Results by Question:")
print(results_df[available_cols].to_string(max_colwidth=60))

# Save results
output_file = 'ragas_evaluation_results.csv'
results_df.to_csv(output_file, index=False)
print(f"\n✓ Results saved to: {output_file}")

# Calculate statistics on valid scores only
print("\nStatistics (excluding NaN/failed evaluations):")
for metric in ['faithfulness', 'context_precision',
               'context_recall']:
    if metric in results_df.columns:
        valid_scores = results_df[metric].dropna()
        if len(valid_scores) > 0:
            print(f"  {metric:20s}: mean={valid_scores.mean():.4f}, "
                  f"min={valid_scores.min():.4f}, max={valid_scores.max():.4f}, "
                  f"valid={len(valid_scores)}/{len(results_df)}")
        else:
            print(f"  {metric:20s}: No valid scores")

print("\n" + "="*70)


                         RAGAS EVALUATION RESULTS

Overall Scores:
  Faithfulness:        [1.0, 1.0, nan, nan, nan, nan, nan, nan, 0.8888888888888888, nan]
  Context Precision:   [0.9999999999, 0.9999999999, nan, nan, 0.9999999999, nan, nan, nan, 0.9999999999, nan]
  Context Recall:      [1.0, 1.0, nan, nan, nan, nan, nan, nan, nan, nan]


Detailed Results by Question:
                                                    user_input  faithfulness  context_precision  context_recall
0  What are the three core pipelines in the FTI (Feature/Tr...      1.000000                1.0             1.0
1  What is an LLM Twin and what is its primary use case in ...      1.000000                1.0             1.0
2  According to the book, what are the main benefits of usi...           NaN                NaN             NaN
3  What are the four main components of the LLM Twin system...           NaN                NaN             NaN
4  What is ZenML and what are its three main features used ...     